# DummyRegressor 기준선(BASELINE) — 전체 센서

이 노트북은 실제 회귀모델과 비교하기 위한 가장 단순한 회귀 기준선입니다.

- 각 LOT에 강종 `GRADE` 연결
- 1초 센서값 → LOT별 1분 평균 집계
- LOT 시간순 Train/Test 분리
- 온도 CSV의 전체 센서 대상
- 각 센서를 하나씩 `y`로 두고 평균 예측
- `DummyRegressor(strategy="mean")`
- Train의 센서 평균값만 계속 예측
- MAE / RMSE / R² 계산
- `CP-Z4M`도 전체 센서 기준선 확인을 위해 포함
- 별도 CSV/TXT 파일 저장 없음


In [ ]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")
FIG_DIR = os.path.join("..", "figures")
os.makedirs(FIG_DIR, exist_ok=True)

TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")

ENC = "utf-8-sig"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 150)

installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in [
    "Malgun Gothic",
    "AppleGothic",
    "NanumGothic",
    "NanumBarunGothic",
    "Noto Sans CJK KR",
]:
    if cand in installed:
        rcParams["font.family"] = cand
        break

rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110


## 1. DummyRegressor와 전체 센서

이번 기준선에서는 센서 간 관계를 학습하지 않습니다.

각 센서를 하나씩 예측 대상 `y`로 두고,
`DummyRegressor(strategy="mean")`이 Train 구간의 센서 평균값만 계속 예측하도록 합니다.


In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

SENSOR_MODELS = {
    "세정존": "CLN-IN",

    "건조존_Z1_TC": "TC-Z1",
    "건조존_Z1_OP": "OP-Z1",
    "건조존_Z2_TC": "TC-Z2",
    "건조존_Z2_OP": "OP-Z2",

    "가열·균열존_Z3_TC": "TC-Z3",
    "가열·균열존_Z3_OP": "OP-Z3",
    "가열·균열존_Z4_TC": "TC-Z4",
    "가열·균열존_Z4_OP": "OP-Z4",
    "가열·균열존_Z5_TC": "TC-Z5",
    "가열·균열존_Z5_OP": "OP-Z5",
    "가열·균열존_Z6_TC": "TC-Z6",
    "가열·균열존_Z6_OP": "OP-Z6",
    "가열·균열존_CP-Z4": "CP-Z4",
    "가열·균열존_CP-Z4M": "CP-Z4M",

    "과시효대_Z7_TC": "TC-Z7",
    "과시효대_Z7_OP": "OP-Z7",
    "과시효대_Z8_TC": "TC-Z8",
    "과시효대_Z8_OP": "OP-Z8",
    "과시효대_Z9_TC": "TC-Z9",

    "냉각대_OUT1": "TC-OUT1",
    "냉각대_OUT2": "TC-OUT2",
}

for sensor_name, sensor_col in SENSOR_MODELS.items():
    print(f"{sensor_name}: {sensor_col}")


## 2. 온도 데이터와 CR1 강종변경 이벤트 불러오기


In [ ]:
temp_df = pd.read_csv(
    TEMP_CSV,
    encoding=ENC,
    parse_dates=["MEAS_DT"],
)

event_df = pd.read_csv(
    EVENT_CSV,
    encoding=ENC,
    parse_dates=["EVT_DT"],
)

temp_df = (
    temp_df
    .sort_values("MEAS_DT")
    .reset_index(drop=True)
)

grade_event = (
    event_df.loc[
        (event_df["PLANT_CD"] == "CR1")
        & (event_df["EVT_TYPE"] == "강종변경"),
        ["EVT_DT", "AFT_VAL"],
    ]
    .sort_values("EVT_DT")
    .copy()
)

print("온도 데이터 행 수:", len(temp_df))
print("CR1 강종변경 이벤트 수:", len(grade_event))


## 3. LOT 시작시각을 이용해 강종(GRADE) 연결


In [ ]:
lot_grade = (
    temp_df
    .groupby("LOT_NO", as_index=False)["MEAS_DT"]
    .min()
    .rename(columns={"MEAS_DT": "LOT_START"})
    .merge(
        grade_event,
        left_on="LOT_START",
        right_on="EVT_DT",
        how="left",
        validate="one_to_one",
    )
    .rename(columns={"AFT_VAL": "GRADE"})
)

if lot_grade["GRADE"].isna().any():
    missing_lots = lot_grade.loc[
        lot_grade["GRADE"].isna(),
        "LOT_NO"
    ].tolist()

    raise ValueError(
        f"강종을 연결하지 못한 LOT가 있습니다: {missing_lots}"
    )

lot_grade = lot_grade[
    ["LOT_NO", "LOT_START", "GRADE"]
]

display(lot_grade)


## 4. 원본 1초 센서값을 LOT별 1분 평균으로 집계

전체 센서를 같은 방식으로 1분 평균 집계합니다.

`CP-Z4M`은 이후 실제 모델 입력에서는 중복 여부에 따라 제외할 수 있지만,
이번 DummyRegressor에서는 **전체 센서의 단순 평균 기준선 확인**이 목적이므로 포함합니다.


In [ ]:
MODEL_SENSOR_COLS = list(SENSOR_MODELS.values())

required_cols = [
    "LOT_NO",
    "MEAS_DT",
] + MODEL_SENSOR_COLS

missing_cols = [
    col
    for col in required_cols
    if col not in temp_df.columns
]

if missing_cols:
    raise ValueError(
        f"온도 CSV에 필요한 컬럼이 없습니다: {missing_cols}"
    )

parts = []

for lot_no, group in temp_df.groupby(
    "LOT_NO",
    sort=False,
):
    group = (
        group
        .sort_values("MEAS_DT")
        .set_index("MEAS_DT")
    )

    rs = group[
        MODEL_SENSOR_COLS
    ].resample("1min")

    minute = rs.mean()

    # 한 분 동안 원본 관측치가 30개 이상인 분만 사용
    minute = minute.loc[
        rs.size() >= 30
    ].copy()

    minute["LOT_NO"] = lot_no

    parts.append(
        minute.reset_index()
    )

base_df = (
    pd.concat(
        parts,
        ignore_index=True,
    )
    .merge(
        lot_grade[
            ["LOT_NO", "GRADE"]
        ],
        on="LOT_NO",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        ["LOT_NO", "MEAS_DT"]
    )
    .reset_index(drop=True)
)

print("1분 집계 행 수:", len(base_df))
print("LOT 수:", base_df["LOT_NO"].nunique())
print("강종 수:", base_df["GRADE"].nunique())
print("센서 수:", len(MODEL_SENSOR_COLS))

display(base_df.head())


## 5. LOT 시간순 Train / Test 분리

현재 14 LOT 기준:
- 앞 11 LOT = Train
- 마지막 3 LOT = Test


In [ ]:
lot_order = (
    lot_grade
    .sort_values("LOT_START")["LOT_NO"]
    .tolist()
)

if len(lot_order) == 14:
    train_lots = lot_order[:11]
    test_lots = lot_order[11:]
else:
    cut = min(
        max(
            3,
            int(len(lot_order) * 0.8),
        ),
        len(lot_order) - 1,
    )

    train_lots = lot_order[:cut]
    test_lots = lot_order[cut:]

base_df["SPLIT"] = np.where(
    base_df["LOT_NO"].isin(
        train_lots
    ),
    "train",
    "test",
)

print("Train LOT:", train_lots)
print("Test LOT :", test_lots)


## 6. 전체 센서 DummyRegressor 기준선 계산

각 센서에서:

- `y = 해당 센서값`
- `DummyRegressor(strategy="mean")`
- `X`는 형식상 필요한 상수값만 사용

DummyRegressor는 다른 센서값을 학습하지 않고,
Train의 해당 센서 평균값을 Test 전체에 동일하게 예측합니다.


In [ ]:
train_mask = base_df["SPLIT"].eq("train")
test_mask = base_df["SPLIT"].eq("test")

dummy_reg_results = []

for sensor_name, sensor_col in SENSOR_MODELS.items():

    print()
    print("=" * 70)
    print(sensor_name)
    print(sensor_col)
    print("=" * 70)

    valid_mask = base_df[sensor_col].notna()

    model_train_mask = (
        train_mask
        & valid_mask
    )

    model_test_mask = (
        test_mask
        & valid_mask
    )

    y_train = base_df.loc[
        model_train_mask,
        sensor_col,
    ]

    y_test = base_df.loc[
        model_test_mask,
        sensor_col,
    ]

    if len(y_train) == 0 or len(y_test) == 0:
        print("유효 Train/Test 데이터가 없어 건너뜁니다.")
        continue

    # DummyRegressor는 X 내용을 사용하지 않지만
    # sklearn의 fit/predict 형식을 맞추기 위해 상수 X를 사용
    X_train = np.zeros(
        (len(y_train), 1)
    )

    X_test = np.zeros(
        (len(y_test), 1)
    )

    dummy = DummyRegressor(
        strategy="mean"
    )

    dummy.fit(
        X_train,
        y_train,
    )

    pred = dummy.predict(
        X_test
    )

    mae = mean_absolute_error(
        y_test,
        pred,
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test,
            pred,
        )
    )

    r2 = r2_score(
        y_test,
        pred,
    )

    predicted_mean = float(
        np.ravel(
            dummy.constant_
        )[0]
    )

    dummy_reg_results.append({
        "구역": sensor_name,
        "센서": sensor_col,
        "TRAIN_ROWS": len(y_train),
        "TEST_ROWS": len(y_test),
        "TRAIN_평균": y_train.mean(),
        "DUMMY_예측값": predicted_mean,
        "TEST_평균": y_test.mean(),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })

    print(
        f"Train 평균={y_train.mean():.4f}"
    )

    print(
        f"Dummy 예측값={predicted_mean:.4f}"
    )

    print(
        f"Test 평균={y_test.mean():.4f}"
    )

    print(
        f"MAE={mae:.4f} | "
        f"RMSE={rmse:.4f} | "
        f"R2={r2:.4f}"
    )


## 7. 전체 센서 DummyRegressor 결과표


In [ ]:
dummy_reg_result_df = pd.DataFrame(
    dummy_reg_results
)

for col in [
    "TRAIN_평균",
    "DUMMY_예측값",
    "TEST_평균",
    "MAE",
    "RMSE",
    "R2",
]:
    dummy_reg_result_df[col] = (
        dummy_reg_result_df[col]
        .round(4)
    )

display(
    dummy_reg_result_df
)
